# Capabilities: choose a question, then a method

Onceover searches for several kinds of structure in the same table. A
finding's category tells you what it states; its `facets` state the same
thing as separate fields, and the category is derived from them — see
[the taxonomy chapter](taxonomy.ipynb). Its evidence identifies the
method that found it. Different methods can support the same statement,
while a single method can produce several kinds of findings.

The standard budget runs the default detector set: 31 of the 39
registered detectors. You can focus the work with `detectors=[...]`,
or declare a numeric response with `target=`. Seven detectors are
opt-in and run only when named: `views` (experimental),
`changeforest`, and `dates`, `pivot`, `period_associations`,
`drift`, and `granularity` — a detector battery measured the last
five producing no findings in 40 runs over 20 datasets, so a
default run records each of them as skipped in coverage instead of
spending time and report space on them. `expectations` and `group_contrast` are in the default set but
engage only on declared hints. This guide starts with user questions; the registry
at the end gives the exact detector IDs and each detector's
activation.

## Can I use this table in a join?

[Keys and duplicates](../methodology/keys_duplicates.ipynb) distinguish a
verified single-column key, a composite key, a nearly unique column, and
repeated rows. These are different properties: a repeated key value need
not identify an exact duplicate row. Counts and final key/duplicate
comparisons cover the full input.

For an extract with repeated orders, mixed invoice formats, and missing
dates, follow the [vendor-extract case study](../case_studies/vendor_extract.ipynb).

## Are values missing or recorded inconsistently?

[Column cards](columns.ipynb) summarize types, nulls, exact distinct counts,
concentration, top values, and character-format patterns. Role suggestions
such as `identifier` or `categorical` are inferences from those counts.

[Sentinels](../methodology/sentinel_values.ipynb) identify repeated,
windowed values that could encode missingness. [Digit fingerprints](../methodology/digit_fingerprints.ipynb)
expose rounding and digit preferences. The `granularity` detector identifies
changes between fractional and integer-valued recording. These findings
point to a production process to inspect, not an automatic recoding rule.

[Extreme-value detection](../methodology/extreme_values.ipynb) finds numeric
values isolated from their column or a categorical subset, and categorical
values that break a nearly universal pattern inside a context. Its evidence
identifies the rows and the reference group.

[Boundary mass](../methodology/boundary_mass.ipynb) reports mass piled at a numeric column's
observed minimum or maximum, and — in a zero-inflated column — an
unusual interior mass point: the pile's count and share against the
counts of its nearest distinct values. The claim states the pile-up
as observed; its interpretation names a recording cap or floor as
one possible cause, not as the conclusion.

A `missingness` pass states the pattern behind a column missing on
at least 1% of rows when one clears its bar — the missing rate
following a grouping level, a numeric threshold, or a point on the
time axis — and blocks of columns missing on the same rows.

## Which columns repeat or derive information?

[Functional dependencies](../methodology/functional_dependencies.ipynb)
check whether equal inputs map to equal outputs. Approximate mappings
report violations. Arithmetic scans verify affine formulas, sums,
differences, products, ratios, and small linear combinations on complete
rows, within a stated tolerance.

The `duplicate_encodings` detector identifies alternative labels for the
same grouping and finer-to-coarser rollups. The `pivot` detector recognizes
a verified long layout and profiles its wide form as well; wide-form
findings are tagged in their evidence. See
[long-format reports](findings.ipynb).

Order constraints state that one column is at most another on every
evaluated row, or all but a counted share of them; a direction is
reported only when it beats random pairing of the two columns. A
row-level pass reports rows that repeat another row on every column
but one, rows far from the line two tightly related columns
otherwise follow, and rows where two columns each hold a value from
the other's range.

A [predictability finding](../methodology/predictability_screen.ipynb)
answers a different question: can a fitted model predict the target on
held-out rows? It can reveal nonlinear structure without establishing an
exact mapping. Its claim names the drivers — the inputs whose share
of the model's split gain clears a per-feature floor — with their
gain shares; when the drivers are a proper subset of the inputs, the
model is refitted on the drivers alone and the claim reports the
refit's holdout score first and the full model's second. Columns
roled as scaffolding (index, identifier, timestamp, and timestamp
parts) are excluded from the predictor pool as well as from the
targets. The [feature-trimming case study](../case_studies/feature_trimming.ipynb)
shows how to separate derivation, predictability, and a merely unique
continuous measurement.

## What happens within a segment?

[PRISM and decision tables](../methodology/rules_tables.ipynb) describe
conditional patterns and compact predictive tables.
[Association mining](../methodology/associations.ipynb) finds pairwise
nominal implications. Reported rules include a common diagnostic recount:
matching rows, success rate, baseline, and lift.

[Hotspots](../methodology/hotspots.ipynb) find combinations whose joint
frequency exceeds what the marginals explain. [Local linear regimes](../methodology/local_linear_regimes.ipynb)
fit different equations in different regions and compare them with a
single global line on held-out rows.

## What shape does a relationship have?

[Pearson, Spearman, and Chatterjee's ξ](../methodology/dependence_measures.ipynb)
describe linear, monotone, and more general pairwise dependence.
[Period associations](../methodology/period_associations.ipynb) can reveal
a sign change hidden by a pooled correlation.

Association statements are tested, not only bar-filtered. Each
screen Benjamini–Hochberg-corrects over the pairs it tested in the
run; a pair is stated as strong when it remains significant and its
effect reaches the 0.5 bar, and as weak when it remains significant
below the bar. Weak findings carry full evidence in the JSON and
collapse into one line in the digest. `report.associations()`
returns every pair a screen computed statistics for, stated or not.
A strong statement with a signed effect also carries a Simpson
reversal check: when the pooled direction reverses inside every
usable stratum of another categorical column, the claim names the
stratifier and states the pooled and within-stratum effects; the
finding itself is never removed or re-tiered.

When the response is known, [target shapes](../methodology/target_shapes.ipynb)
compare a line, a binned curve, and a single step. The
[interaction pass](../methodology/interactions.ipynb) asks whether two
predictors act non-additively in a fitted model of a target. An interaction
is not the same as association between the two predictors themselves.

The experimental [views detector](../methodology/view_clusters.ipynb)
asks which columns can share a fitted clustering of rows. Its
[method reference](../methodology/view_clusters_reference.ipynb) explains
the score, search, and additional cluster/density findings.

## Where does behavior change?

[Temporal analysis](../methodology/temporal_analysis.ipynb) covers inferred
calendar gaps, irregular intervals, ordering, appended backfills, sustained
level/trend/variance changes, and transient excursions. A
[residual shift](../methodology/local_linear_regimes.ipynb) measures change
after accounting for a stated reference model.

[Discontinuities](../methodology/step_discontinuities.ipynb) find threshold
jumps across numeric axes, including diagonal boundaries, without
requiring time order. [Drift](../methodology/drift.ipynb) compares how later
periods route through an early-period model. Read its scores as changes in
model-used predictor distributions, not a general test of target behavior.

[Declared expectations](../methodology/expectations.ipynb) check a supplied
calendar, complete sequence, lexicographic ordering, or panel layout.
They distinguish observed irregularity from a violation of a stated
expectation.

## Select the work explicitly

This small fixture has a nonlinear amount. A target-aware run adds a shape
description to the normal profile. An explicit detector list instead
replaces the default selection:


In [1]:
import numpy as np
import polars as pl
import onceover

rng = np.random.default_rng(42)
x = rng.uniform(-1, 1, 800)
df = pl.DataFrame({"driver": x, "amount": 100 + 30 * x * x + rng.normal(0, 0.2, len(x))})

shaped = onceover.profile(df, target="amount")
for finding in shaped.select(category="dependencies", target="amount"):
    print(finding["specification"].get("form"), finding["claim"])

focused = onceover.profile(df, detectors=["numeric_dependence", "identities"])
print("Focused run:")
print(focused.to_digest(1_200))


None amount is predictable from driver (100% of gain) (holdout R² = 0.99)
curved_shape driver relates to amount as a non-monotone curve, not a line: a binned curve explains 98.6% of amount's variance, against 0.3% for a straight line (Spearman -0.04)
Focused run:
profile digest: 800 rows x 2 columns, 1 findings, schema 0.7
headline: 800 rows × 2 columns (standard budget); 0 functional dependencies, 0 predictive dependencies, 0 conditional rules
coverage: association_rules=skipped, boundaries=skipped, breaks=skipped, changeforest=skipped, content=skipped, dates=skipped, decision_tables=not_applicable, digits=skipped, discontinuities=skipped, drift=skipped, duplicate_encodings=skipped, expectations=skipped, functional_dependencies=skipped, granularity=skipped, group_contrast=skipped, hotspots=skipped, interactions=skipped, keys=not_applicable, missingness=skipped, order_constraints=skipped, ordered_partitions=skipped, panel_coverage=skipped, period_associations=skipped, pivot=skipped, re

With an explicit detector list, include `target_shapes` if you want shape
analysis. Required resource detectors can run without contributing their
own findings. `quick` omits model-based relationship searches; larger
budgets widen the methods' own limits. See [coverage](coverage.ipynb) and
[the API reference](api.ipynb) for exact controls.

## Detector registry

The following registry comes from the installed package. The purpose column
connects each ID to its role in the guide. The activation column states how
a detector runs: `default` (in the default set), `opt_in` (only when named
in `detectors=[...]`), `hint_gated` (in the default set, engages only on
declared hints), or `target_gated` (runs when a target is designated). A registered detector can still
be ineligible for a particular input, require a declared target/time axis,
or be omitted by the budget. Read its coverage rather than inferring work
from its presence in the registry.


In [2]:
from great_tables import GT, loc, style

purposes = {
    "column_profile": "Column summaries, role suggestions, missingness, concentration, string formats",
    "row_duplicates": "Repeated rows under the stated comparison and missing-value policy",
    "keys": "Single-column, composite, and nearly unique keys",
    "pivot": "Verified long layout and relationships in its wide form",
    "functional_dependencies": "Exact or approximate value mappings, including multi-column determinants",
    "order_constraints": "Pairwise order between two columns (a <= b), exact or with counted violations",
    "duplicate_encodings": "Alternative labels for a grouping and finer-to-coarser rollups",
    "sortedness": "Order in the row sequence and late appended backfills",
    "granularity": "Changes between fractional and integer-valued recording",
    "sentinels": "Repeated, windowed candidate missing-value codes",
    "boundaries": "Mass piled at a numeric column's minimum or maximum, and interior mass points in zero-inflated columns",
    "missingness": "Each column's missing rate, the grouping level, threshold, or time point it follows, and columns missing on the same rows",
    "content": "What text columns hold (numbers, dates, recognized formats), label variants and likely misspellings, repeated extremes, stray signs, zero inflation, separated modes, and far-out dates",
    "series": "Absent steps on the time axis; trend, cycle and persistence of each measure; grouping levels that appear, end or shift share over time",
    "row_consistency": "Rows repeating another on every column but one, rows far from the line two tight columns follow, and rows with two columns' values swapped",
    "digits": "Heaping, terminal-digit preferences, and eligible Benford deviations",
    "dates": "Inferred calendar gaps and interval irregularity",
    "expectations": "Declared sequences, calendars, ordering, and panel completeness",
    "panel_coverage": "Per-period row and entity volume on the time axis, against the typical period",
    "group_contrast": "How one declared subset of the rows differs from the rest of the table",
    "gbm": "Predictability of eligible columns from the other columns",
    "decision_tables": "Compact predictive tables selected by cross-validation",
    "prism": "Multi-condition rules for categorical targets",
    "regimes": "Local linear equations and dated reference-model residual shifts",
    "interactions": "Non-additive predictor pairs under a fitted target model",
    "target_shapes": "Line, curve, and step comparisons for a designated numeric target",
    "breaks": "Sustained level, trend, variance changes, and transient excursions",
    "ordered_partitions": "Recursive model-based partitions of a target over time and dimension splits",
    "surprises": "Isolated numeric values and context-breaking categorical values",
    "discontinuities": "Axis-aligned and diagonal threshold jumps",
    "drift": "Reference-period model-routing changes along a declared time axis",
    "changeforest": "Distribution changes in the measures jointly along the time axis, from classifier-based changepoint detection",
    "ordered_axis": "Shape of a response along an ordered axis (monotonicity, reversal blocks, saturation) and located level, trend, slope, regime, and spike events along it, including inside one categorical level",
    "association_rules": "Pairwise nominal rules with train and diagnostic evidence",
    "hotspots": "Combinations concentrated beyond their marginal frequencies",
    "numeric_dependence": "Pearson, Spearman, and both directions of Chatterjee's xi",
    "identities": "Affine and small multi-term arithmetic relationships",
    "period_associations": "Changes in pairwise correlation across calendar or year periods",
    "ordered_partitions": "Model-based recursive partitions of a numeric response along a time axis or a categorical dimension",
    "views": "Experimental shared row partitions, cluster differences, and density findings",
}
registry = onceover.available_detectors()
assert set(purposes) == {d["id"] for d in registry}
registry_rows = [{"Detector": d["id"], "Purpose": purposes[d["id"]],
                  "Activation": d["activation"],
                  "Resource dependencies": ", ".join(d["requires"]) or "None"}
                 for d in registry]
(GT(pl.DataFrame(registry_rows))
 .tab_header(title="Available detectors", subtitle=f"Installed onceover {onceover.__version__}")
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


GT(_tbl_data=┌─────────────────────────┬─────────────────────────────────┬────────────┬───────────────────────┐
│ Detector                ┆ Purpose                         ┆ Activation ┆ Resource dependencies │
╞═════════════════════════╪═════════════════════════════════╪════════════╪═══════════════════════╡
│ column_profile          ┆ Column summaries, role suggest… ┆ default    ┆ None                  │
│ row_duplicates          ┆ Repeated rows under the stated… ┆ default    ┆ None                  │
│ keys                    ┆ Single-column, composite, and … ┆ default    ┆ None                  │
│ pivot                   ┆ Verified long layout and relat… ┆ opt_in     ┆ None                  │
│ functional_dependencies ┆ Exact or approximate value map… ┆ default    ┆ None                  │
│ …                       ┆ …                               ┆ …          ┆ …                     │
│ hotspots                ┆ Combinations concentrated beyo… ┆ default    ┆ None                  │
│ numeric_dependence      ┆ Pearson, Spearman, and both di… ┆ default    ┆ None                  │
│ identities              ┆ Affine and small multi-term ar… ┆ default    ┆ numeric_dependence    │
│ period_associations     ┆ Changes in pairwise correlatio… ┆ opt_in     ┆ numeric_dependence    │
│ views                   ┆ Experimental shared row partit… ┆ opt_in     ┆ None                  │
└─────────────────────────┴─────────────────────────────────┴────────────┴───────────────────────┘, _body=<great_tables._gt_data.Body object at 0x7ed934127c80>, _boxhead=Boxhead([ColInfo(var='Detector', type=<ColInfoTypeEnum.default: 1>, column_label='Detector', column_align='left', column_width=None, stub_level=None), ColInfo(var='Purpose', type=<ColInfoTypeEnum.default: 1>, column_label='Purpose', column_align='left', column_width=None, stub_level=None), ColInfo(var='Activation', type=<ColInfoTypeEnum.default: 1>, column_label='Activation', column_align='left', column_width=None, stub_level=None), ColInfo(var='Resource dependencies', type=<ColInfoTypeEnum.default: 1>, column_label='Resource dependencies', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x7ed905ad67b0>, _spanners=Spanners([]), _heading=Heading(title='Available detectors', subtitle='Installed onceover 0.1.5', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x7ed9056dfd40>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x7ed928128fe0>, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=0, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=1, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=2, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=3, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=4, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=5, colnum=None, styles=[CellStyleCss(rule='white-space: normal; overflow-wrap: anywhere; line-height: 1.5;')]), StyleInfo(locname=LocBody(columns=None, rows=None, mask=None), grpname=None, colname='Detector', rownum=6, colnum=None, styles=